# Prediction on all yellow fever simulated reads (no storage)

This is a a reference notebook for prediction inference on yellow fever simulated reads, using the a finetuned model, on simulated reads generated from all reference sequences.

- Simulated reads from an aligned file generated by ART Illumina simulator (`*.aln file`).
- Uses the dataset class `AlnFileDataset` to read batches of simulated reads and their metadata.
- Uses the CNN Virus finetune model to predict the label and position probabilities and classes for each simreads.
- Creates a prediction report save data for analysis.

> **Note**: 
>
> When an `*.aln` file counts a very large number of simulated reads, running a prediction on all of them is very time consuming. In such case, we can use the filter feature of `AlnFileDataset` to select a specific reference sequence ID (`refseqid`) to filter on, i.e. the dataset will only yield data (read sequence, label, position and optionaly metadata) for those reads generated from that reference sequence.

# 1. Imports and setup environment

In [ ]:
# Install required custom packages if not installed yet.
import importlib.util
if not importlib.util.find_spec('eccore'):
    print('installing package: `eccore`')
    ! pip install -qqU eccore
else:
    print('`eccore` already installed')
if not importlib.util.find_spec('metagentorch'):
    print('installing package: `metagentorch')
    ! pip install -qqU metagentorch
else:
    print('`metagentorch` already installed')

`eccore` already installed
`metagentorch` already installed


In [ ]:
# Import all required packages
import os
from datetime import datetime
from functools import partial
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, Generator, List, Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shutil
from eccore.core import files_in_tree
from eccore.ipython import nb_setup
from nbdev import show_doc
from tqdm.notebook import tqdm, trange

# Setup the notebook for development
nb_setup()

os.environ['KERAS_BACKEND'] = "torch"
import keras
import torch
from torch.utils.data import DataLoader

from metagentorch.core import ProjectFileSystem, list_available_devices
from metagentorch.cnn_virus.architecture import create_model_original
from metagentorch.cnn_virus.data import (AlnFileDataset, AlnFileReader,
                                         OriginalLabels,
                                         combine_predictions,
                                         split_kmer_batch_into_50mers)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Set autoreload mode


List all computing devices available on the machine

In [ ]:
list_available_devices()

CUDA available: True
Number of CUDA devices: 1
CUDA Device 0: NVIDIA GeForce GTX 1050
CPU available: cpu


# 2. Setup paths to files

Key folders and system information

In [ ]:
pfs = ProjectFileSystem()
pfs.info()

Running linux on local computer
Device's home directory: /home/vtec
Project file structure:
 - Root ........ /home/vtec/projects/bio/metagentorch 
 - Data Dir .... /home/vtec/projects/bio/metagentorch/data 
 - Notebooks ... /home/vtec/projects/bio/metagentorch/nbs


Set the path to the pretained model and the virus labels mapping file:

- `p2model`: path to file with the selected saved finetuned model
- `p2virus_labels` path to file with virus names and labels mapping for original model

In [ ]:
p2finetuned = pfs.data / 'saved/cnn_virus_finetuned'
files_in_tree(p2finetuned);

saved
  |--cnn_virus_finetuned
  |    |--yfv_5seq_lr_0.0001_epochs_25_20250303_093813.weights.h5 (0)
  |    |--yfv_5seq_2025-03-05_07h41m_7a534024-be3c-47e9-8323-9caeb3febaa7.weights.h5 (1)
  |    |--_model_stats.json (2)
  |    |--yfv_5seq_lr_0.0001_epochs_25_20250303_233339.weights.h5 (3)
  |    |--training_stats
  |    |--training_curves
  |    |    |--yfv_5seq_2025-03-05_07h41m_7a534024-be3c-47e9-8323-9caeb3febaa7.curves.png (4)


In [ ]:
ft_model_fname = 'yfv_5seq_2025-03-05_07h41m_7a534024-be3c-47e9-8323-9caeb3febaa7.weights.h5'

p2model = p2finetuned / ft_model_fname
assert p2model.is_file(), f"No file found at {p2model.absolute()}"

p2virus_labels = pfs.data / 'CNN_Virus_data/virus_name_mapping'
assert p2virus_labels.is_file(), f"No file found at {p2virus_labels.absolute()}"

Set the path to the simulated read we want to use and review the content.

In [ ]:
fnames = files_in_tree(pfs.data / 'ncbi/simreads/yf', pattern='69seq')

simreads
  |--yf
  |    |--single_69seq_150bp
  |    |    |--single_69seq_150bp.fq (0)
  |    |    |--single_69seq_150bp.aln (1)
  |    |    |--single_69seq_150bp_list_reads.json (2)
  |    |--single_finetune_5seq_50bp_2025-03-04_22h16m


In [ ]:
file_stem = 'single_69seq_150bp'

p2aln = pfs.data / f"ncbi/simreads/yf/{file_stem[:-2] if file_stem[-1] in ['1', '2'] else file_stem}/{file_stem}.aln"
assert p2aln.exists()

aln = AlnFileReader(p2aln)

print(f"Reading alignment file: {p2aln.name}:\n")
for i, aln_read in enumerate(aln):
    pass

print(f"  - {i+1:,d} simulated reads in file '{p2aln.name}' from {len(aln.header['reference sequences'])} reference sequences.")
print('  - ART Illumina command: ',aln.header['command'])
print('  - Reference Sequences:')
print('     ','\n      '.join(aln.header['reference sequences']))

Reading alignment file: single_69seq_150bp.aln:

  - 1,160,343 simulated reads in file 'single_69seq_150bp.aln' from 69 reference sequences.
  - ART Illumina command:  /usr/bin/art_illumina -i /home/vtec/projects/bio/metagentools/data/ncbi/refsequences/yf/yf_2023_yellow_fever.fa -ss HS25 -l 150 -f 250 -o /home/vtec/projects/bio/metagentools/data/ncbi/simreads/yf/single_69seq_150bp/single_69seq_150bp -rs 1731415434
  - Reference Sequences:
      @SQ	11089:ncbi:1	1	AY968064	11089	ncbi	Angola_1971	10234
      @SQ	11089:ncbi:2	2	U54798	11089	ncbi	Ivory_Coast_1982	10231
      @SQ	11089:ncbi:3	3	DQ235229	11089	ncbi	Ethiopia_1961	10234
      @SQ	11089:ncbi:4	4	AY572535	11089	ncbi	Gambia_2001	10231
      @SQ	11089:ncbi:5	5	MF405338	11089	ncbi	Ghana_Hsapiens_1927	10231
      @SQ	11089:ncbi:6	6	U21056	11089	ncbi	Senegal_1927	10231
      @SQ	11089:ncbi:7	7	AY968065	11089	ncbi	Uganda_1948	10234
      @SQ	11089:ncbi:8	8	JX898871	11089	ncbi	ArD114896_Senegal_1995	10231
      @SQ	11089:ncbi:9	9	JX898

# 3. Inference Loop

## Utility Functions

In [ ]:
def top_predictions(probs, n=5):
    """Returns the top n top predictions for each kmer read"""

    def top_n_most_frequent(preds, n=5):
        """Returns the top n most frequent predictions for each 50read"""
        # print(preds.shape)
        uniques, counts = np.unique(preds, return_counts=True)
        top_idx = np.argsort(counts)[-n:]
        return uniques.take(top_idx)

    top_preds_in_50mers = np.argsort(probs, axis=-1)[:, :, -n:]
    nb_kmers, nb_50mers, nb_lbls = top_preds_in_50mers.shape
    top_preds_in_kmer = top_preds_in_50mers.reshape(nb_kmers,nb_50mers * nb_lbls)

    return np.apply_along_axis(top_n_most_frequent, axis=1, arr=top_preds_in_kmer, n=n)

In [ ]:
def count_successive_label_preds(
    label_50mer_probs: torch.Tensor   # Batch of 50-mer label probs (batch_size, k-49, nb_label_class)
    ) -> tuple[torch.Tensor, List[List[int]], List[Dict[int, List[int]]]]:
    """Counts succcession of identical 50-mer predictions for each kmer

    Returns:
    - label_50mer_preds_per_kmer is a tensor with the all 50-mer reads label prediction each k-mer (shape: [bs, k-49])
    - flat_series is a list bs lists (one per k-mer) providing all successions of identical predictions
    - series is a dictionary with:
        - keys equal to each of the possible labels
        - values as a list of the length of each succession of identical prediction for that label
    """
    b = label_50mer_probs.shape[0]
    k = 150
    series = []
    flat_series = []
    label_50mer_preds_per_kmer = torch.argmax(label_50mer_probs, dim=2)  # shape [bs, k-49]
    same_as_next = torch.roll(label_50mer_preds_per_kmer, shifts=-1, dims=1) == label_50mer_preds_per_kmer
    same_as_previous = torch.roll(label_50mer_preds_per_kmer, shifts=+1, dims=1) == label_50mer_preds_per_kmer
    xor = torch.bitwise_xor(same_as_next,same_as_previous)
    for i in range(b):
        kmer_series_idxs = torch.arange(k-49)[xor[i]]
        start_idxs = kmer_series_idxs[0::2]
        end_idxs = kmer_series_idxs[1::2]
        flat_series.append((end_idxs-start_idxs+1).numpy().tolist())
        kmer_series = {key: [] for key in range(187)}
        for s,e in zip(start_idxs, end_idxs):
            kmer_series[label_50mer_preds_per_kmer[i,s].numpy().item()].append((e-s+1).numpy().item())
        series.append(kmer_series)
    return label_50mer_preds_per_kmer, flat_series, series


In [ ]:
def successive_preds_score(
    label_50mer_preds_per_kmer: torch.Tensor,   # Batch of 50-mer label probs (batch_size, k-49)
    flat_series, 
    true_label=118
    ) -> torch.Tensor:  # Successive Prediction Score
    """
    Evaluate score S(preds, true_label) for a list or predictions and the target value:
        S = (C / N) * (1 + (M / N) * (1 - F / C))
        Where:
        - N is the total number of elements in the list
        - C is the count of elements equal to y
        - M is the maximum length of any continuous sequence of y in the list
        - F is the number of "fragments" or separate sequences of y in the list
    """
    b, k = label_50mer_preds_per_kmer.shape
    nb_50mer = k - 49
    label_50mer_preds_per_kmer = label_50mer_preds_per_kmer.float()
    N = torch.tensor([nb_50mer] * b, dtype=torch.float32)
    C = torch.sum((label_50mer_preds_per_kmer == true_label).float(), dim=1)
    C_safe = C + torch.tensor([0.0001]*b, dtype=torch.float32)
    M = torch.tensor([max(l) if len(l)>0 else 0 for l in flat_series], dtype=torch.float32)
    F = torch.tensor([len(l) for l in flat_series], dtype=torch.float32)
    score = (C / N) * (1 + (M / N) * (1 - F / C_safe))
    return score

In [ ]:
def probabilities_scores(
    probs: torch.Tensor,              # Probabilities for the labels classes for each 50-mer (shape: [bs, k-49,187])
    sequence_threshold: float = 0.9,   # Threshold to consider a prediction as valid
    sequence_weight: float = 0.3,      # Weight for the sequence bonus in the final score
    true_label: int = 118             # The true label to consider
    ) -> torch.Tensor:                # Score for each read (shape: [bs])
    """ Score the 50-mer probabilities for each reads as follows:

        - the ideal list is a list whose elements are all 1
        - a list with many elements close to 1 gets a good score
        - between two lists, with the same elements but in different orders, 
          the one with the longest sequence of numbers closed to 1 gets a better score 

    The score is evaluated as a combination of:
        - Average proximity of the probabilities to 1
        - A longest sequence bonus
        - Weighted scoring between the two metrics above
    How It Works?
        - Average proximity to 1: 
          This is simply the mean of all probabilities in the list. 
          It ensures that lists with many elements close to 1 get a good score.
        - Longest sequence bonus: 
          This part finds the longest continuous sequence of probabilities above a certain threshold (default 0.9). 
          It's then normalized by dividing by the list length.
        - Weighted scoring: 
          The final score is a weighted combination of the average proximity and the sequence bonus. 
          You can adjust the sequence_weight to give more or less importance to the sequence aspect.
    """
    
    b = probs.shape[0]
    k = 150
    nb_50mer = k - 49
    sequence_weight_tensor = torch.tensor([sequence_weight]*b, dtype=torch.float32)

    # 1. Calculate average proximity to 1
    true_label_probs = probs[:, :, true_label] # get the probabilities for the true label
    base_score = torch.mean(true_label_probs, dim=1)

    # 2. Find the longest sequence of numbers above the threshold
    prob_above_threshold = (true_label_probs >= sequence_threshold).float()

    flat_series = []
    same_as_next = torch.roll(prob_above_threshold, shifts=-1, dims=1) == prob_above_threshold
    same_as_previous = torch.roll(prob_above_threshold, shifts=1, dims=1) == prob_above_threshold
    xor = torch.bitwise_xor(same_as_next.long(), same_as_previous.long())
    
    for i in range(b):
        kmer_series_idxs = torch.arange(k-49)[xor[i].bool()]
        start_idxs = kmer_series_idxs[0::2]
        end_idxs = kmer_series_idxs[1::2]
        flat_series.append((end_idxs-start_idxs+1).tolist())

    max_sequence_lengths = torch.tensor([max(l) if len(l)>0 else 0 for l in flat_series], dtype=torch.float32)

    # Normalize the sequence bonus
    sequences_bonus = max_sequence_lengths / k

    # 3. Combine the scores with weighting
    final_scores = (torch.ones_like(base_score)-sequence_weight_tensor) * base_score + sequence_weight_tensor * sequences_bonus
    
    return final_scores

## Inference for all selected refsequences

Define the inference loop for one selected refseqid

In [ ]:
def infer_one_refseq(
    refseqid: str,         # the id of the reference sequence from which reads were simulated
    aln,                   # AlnFileReader object for the aln file containing all the reads
    params,                # dictionary of parameters for the inference
    p2infer_results: Path, # path to directory store the inference results
     ) ->   Tuple[Optional[torch.Tensor], Optional[List[List[int]]], Optional[torch.Tensor], Optional[torch.Tensor]]: # label_probs_kmer, flat_series, preds_scores, probs_scores
    """Run inference on reads from a specific reference sequence.
    
    Args:
        refseqid: The ID of the reference sequence
        aln: AlnFileReader object containing the reads
        params: Dictionary of parameters for inference
        p2infer_results: Path to store inference results
        
    Returns:
        Tuple containing:
        - label_probs_kmer: Tensor of label probabilities for each k-mer
        - flat_series: List of series lengths for each read
        - preds_scores: Tensor of prediction scores
        - probs_scores: Tensor of probability scores
    """
    
    # Data parameters           
    b = params.get('b', 64)     # number of k-mer in a batch
    k = params.get('k', 150)    # read length
    true_label = params.get('true_label', 118)  # yellow fever virus
    top_n = params.get('top_n', 5)              # n for top-n prediction to keep

    print(f"Run prediction loop with the following parameters:")
    print(f"   refseqid: {refseqid}")
    print(f"   {b} k-mer per batch; {k} bp per sequence; keep top-{top_n} predictions")

    # Inference loop parameters
    run_all_batches = params.get('run_all_batches', True)
    nb_batches_to_run = params.get('nb_batches_to_run', 2)

    #====================================================================================================
    # Setup prediction Loop
    #====================================================================================================
    nb_50mer = k - 49
    uid = datetime.today().strftime('%Y-%m-%d_%H_%M_%S')

    aln.reset_iterator()
    model = create_model_original(path2parameters=p2model)
    print(f"Model loaded and ready to run ...")

    # Create list of columns for prediction and probabilities reports
    pred_cols_str = 'readid refseqid refsource refseq_strand taxonomyid'.split(' ')
    pred_cols_int = 'lbl_true lbl_pred pos_true pos_pred'.split(' ')
    top_pred_cols = [f"top_{top_n}_lbl_pred_{i}" for i in range(top_n)]
    score_cols = ['preds_score', 'probs_score']

    # Storage parameters
    # p2store = pfs.data / f"ncbi/infer_results/yf-ncbi/local-inf/{refseqid.replace(':','_')}.csv"
    p2store = p2infer_results/f"{refseqid.replace(':','_')}.csv"
    p2store.unlink(missing_ok=True)
    with open(p2store, 'w') as f:
        f.write(','.join(pred_cols_str + pred_cols_int + top_pred_cols + score_cols) + '\n')

    def tprint(string):
        print(f"{datetime.now().strftime('%H:%M:%S')}    {string}")

    #====================================================================================================
    # Setup prediction Loop
    #====================================================================================================
    print(f"Run prediction loop with the following parameters:")
    print(f"   refseqid: {refseqid}")
    print(f"   {b} k-mer per batch; {k} bp per sequence; keep top-{top_n} predictions")
    tprint(f"Starting prediction loop ...")
    ds = AlnFileDataset(p2file=p2aln,label=true_label, return_metadata=True)
    dl = DataLoader(ds, batch_size=b, shuffle=False)

    # Proceed with prediction inference 
    label_probs_kmer, flat_series, preds_scores, probs_scores = None, None, None, None

    batch_nb = 0
    for i, (read_kmer_b, (label_kmer_b, position_kmer_b), metadata_batch) in enumerate(dl):
        # skip any reference sequence not in the selected list
        any_selected_refseq_in_batch = any([rsid == refseqid for rsid in metadata_batch['refseqid']])
        if not any_selected_refseq_in_batch:
            tprint(f"Skipping batch {i+1:,d} because does not include any read from {refseqid}")
            continue
        batch_nb += 1
        loop_start = datetime.now()

        # reads_kmer, (labels_true, position_true) = string_input_batch_to_tensors(reads_batch, k=k)
        reads_50mer, _ = split_kmer_batch_into_50mers(read_kmer_b)
        assert reads_50mer.shape == ((read_kmer_b.shape[1]-49) * b, 50, 5), f"Problem with shape in batch {i+1}: {reads_50mer.shape}"

        tprint(f'  Starting prediction for {b:,} kmer reads ...')
        label_probs, position_probs = model.predict(reads_50mer)

        tprint('  Reshaping predictions ...')
        label_probs_kmer = torch.from_numpy(label_probs).reshape(b, nb_50mer, -1)
        position_probs_kmer = torch.from_numpy(position_probs).reshape(b, nb_50mer, -1)

        tprint('  Combining predictions ...')
        label_predictions, position_predictions = combine_predictions(label_probs_kmer, position_probs_kmer)
        top_preds = top_predictions(label_probs_kmer, n=top_n)
        
        label_preds_per_kmer, flat_series, series = count_successive_label_preds(label_probs_kmer)
        preds_scores = successive_preds_score(label_preds_per_kmer, flat_series, true_label=true_label)
        probs_scores = probabilities_scores(label_probs_kmer)

        # Add results for current batch
        tprint('  Preparing prediction report ...')
        preds_report = np.concatenate(
            [
                np.expand_dims(np.array(metadata_batch['readid']), axis=1),         # readid 
                np.expand_dims(np.array(metadata_batch['refseqid']), axis=1),       # refseqid
                np.expand_dims(np.array(metadata_batch['refsource']), axis=1),      # refsource
                np.expand_dims(np.array(metadata_batch['refseq_strand']), axis=1),  # refseq_strand
                np.expand_dims(np.array(metadata_batch['reftaxonomyid']), axis=1),  # taxonomyid
                np.expand_dims(np.array([true_label]*b), axis=1),                   # lbl_true
                np.expand_dims(label_predictions.numpy(), axis=1),                  # lbl_pred
                np.expand_dims(np.array(metadata_batch['aln_start_pos']), axis=1),  # pos_true
                np.expand_dims(position_predictions.numpy(), axis=1),               # pos_pred
                top_preds[:, ::-1],                                                 # top_5_lbl_pred_0, top_5_lbl_pred_1, top_5_lbl_pred_2, top_5_lbl_pred_3, top_5_lbl_pred_4
                np.expand_dims(preds_scores.numpy(), axis=1),                       # int_score
                np.expand_dims(probs_scores.numpy(), axis=1)                        # float_score
            ],
            axis=1
        )

        df_preds = pd.DataFrame(
            data=preds_report, 
            columns=pred_cols_str + pred_cols_int + top_pred_cols + score_cols
            )
        tprint('  Saving batch prediction report ...')
        df_preds.to_csv(p2store, mode='a', index=False, header=False)

        tprint(f"  Batch processing time: {(datetime.now() - loop_start).total_seconds():.2f} sec")
        if not run_all_batches and batch_nb >= nb_batches_to_run: 
            print('Stopping')
            break

    print(f"All batches done for {refseqid} ...")

    return label_probs_kmer, flat_series, preds_scores, probs_scores

Define the parameters and run the loop for all selected sequences.

Inference report is saved in `data/ncbi/infer_results/yf-ncbi/local-inf/refseqid.csv`

In [ ]:
p2reports = pfs.data / 'ncbi/infer_results/yf-ncbi/local-inf/' / f"model-ft_{p2model.stem.split('.')[0]}"
assert p2reports.is_dir()
print(f"Will save reports for each reference sequence in:\ndata/ncbi/infer_results/yf-ncbi/local-inf/{p2reports.name}")

Will save reports for each reference sequence in:
data/ncbi/infer_results/yf-ncbi/local-inf/model-ft_yfv_5seq_2025-03-05_07h41m_7a534024-be3c-47e9-8323-9caeb3febaa7


List last 3 generated report files in the directory, with respective file size and nbr samples in order to check whether one of these was not completed.

In [ ]:
reports_by_modification_date = [p for p in p2reports.iterdir() if p.is_file() and p.name.endswith('.csv')]
reports_by_modification_date.sort(key=lambda x: x.stat().st_mtime, reverse=True)
print(f"{'Report':^24s}|{'Number samples':^24s}|{'Modified on':^24s}")
print(f"{'-'*24}|{'-'*24}|{'-'*24}")
for p in reports_by_modification_date[:5]:
    df = pd.read_csv(p)
    print(f"{p.name:^24s}|         {df.shape[0]-1:,}         |  {datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')}")

         Report         |     Number samples     |      Modified on       
------------------------|------------------------|------------------------
   11089_ncbi_69.csv    |         13,599         |  2025-04-27 08:23:40
   11089_ncbi_68.csv    |         15,299         |  2025-04-27 07:50:37
   11089_ncbi_67.csv    |         13,599         |  2025-04-27 07:13:42
   11089_ncbi_66.csv    |         18,699         |  2025-04-27 06:39:54
   11089_ncbi_65.csv    |         18,699         |  2025-04-27 05:57:02


If one of these report files seems inconplete, delete.

In [ ]:
# Existing reports by modification date, last is the newest
report_files_by_modification_date = sorted(p2reports.glob('*.csv'), key=lambda f: f.stat().st_mtime)
refseq_metadata = aln.parse_header_reference_sequences()
existing_reports = [p.stem.replace('_', ':') for p in p2reports.glob('*.csv')]
selected_refseqs = [refseqid for refseqid in refseq_metadata.keys() if refseqid not in existing_reports]
print(len(selected_refseqs), 'refseq remaining')
print(selected_refseqs) 

0 refseq remaining
[]


In [ ]:
params = {
        # Data parameters           
    # 'b': 850,           # number of k-mer in a batch 850 to cover a dividor of a set of one refseq (17000)
    'b': 1700,          # number of k-mer in a batch 850 to cover a dividor a set of one refseq (17000)
    'k': 150,           # read length
    'true_label': 118,  # yellow fever virus
    'top_n': 5,         # n for top-n prediction to keep
    'run_all_batches': True,
    'nb_batches_to_run': 2
}

# for refseqid in selected_refseqs[0:1]:
for refseqid in selected_refseqs:
    print(refseqid)
    label_probs_kmer, flat_series, preds_scores, probs_scores = infer_one_refseq(refseqid, aln, params, p2reports)

## Verify inference reports

Check that each report file includes only inference for reads simulated from the one reference sequence.

In [ ]:
report_files= sorted(p2reports.glob('*.csv'), key=lambda f: int(f.stem.split('_')[-1]))
print([p.name for p in report_files])

['11089_ncbi_1.csv', '11089_ncbi_2.csv', '11089_ncbi_3.csv', '11089_ncbi_4.csv', '11089_ncbi_5.csv', '11089_ncbi_6.csv', '11089_ncbi_7.csv', '11089_ncbi_8.csv', '11089_ncbi_9.csv', '11089_ncbi_10.csv', '11089_ncbi_11.csv', '11089_ncbi_12.csv', '11089_ncbi_13.csv', '11089_ncbi_14.csv', '11089_ncbi_15.csv', '11089_ncbi_16.csv', '11089_ncbi_17.csv', '11089_ncbi_18.csv', '11089_ncbi_19.csv', '11089_ncbi_20.csv', '11089_ncbi_21.csv', '11089_ncbi_22.csv', '11089_ncbi_23.csv', '11089_ncbi_24.csv', '11089_ncbi_25.csv', '11089_ncbi_26.csv', '11089_ncbi_27.csv', '11089_ncbi_28.csv', '11089_ncbi_29.csv', '11089_ncbi_30.csv', '11089_ncbi_31.csv', '11089_ncbi_32.csv', '11089_ncbi_33.csv', '11089_ncbi_34.csv', '11089_ncbi_35.csv', '11089_ncbi_36.csv', '11089_ncbi_37.csv', '11089_ncbi_38.csv', '11089_ncbi_39.csv', '11089_ncbi_40.csv', '11089_ncbi_41.csv', '11089_ncbi_42.csv', '11089_ncbi_43.csv', '11089_ncbi_44.csv', '11089_ncbi_45.csv', '11089_ncbi_46.csv', '11089_ncbi_47.csv', '11089_ncbi_48.csv', 

In [ ]:
reports_to_modify = []
for p in report_files:
    refseqid = p.stem.replace('_',':')
    df = pd.read_csv(p)
    to_modify = df.loc[df.refseqid!=refseqid]
    if len(to_modify) > 0:
        shutil.copy(p, p.parent/f"{p.stem}-initial.csv")
        reports_to_modify.append(p)
        print(f"{refseqid:15s}: NOK {df.shape} {len(to_modify):,d} rows to modify {to_modify.refseqid.unique().tolist()}")
        df = df.loc[df.refseqid==refseqid]
        df.to_csv(p, index=False)
    else:
        print(f"{refseqid:15s}: OK  {df.shape}")

11089:ncbi:1   : OK  (17000, 16)
11089:ncbi:2   : OK  (17000, 16)
11089:ncbi:3   : OK  (17000, 16)
11089:ncbi:4   : OK  (17000, 16)
11089:ncbi:5   : OK  (17000, 16)
11089:ncbi:6   : OK  (17000, 16)
11089:ncbi:7   : OK  (17000, 16)
11089:ncbi:8   : OK  (17000, 16)
11089:ncbi:9   : OK  (17000, 16)
11089:ncbi:10  : OK  (17000, 16)
11089:ncbi:11  : OK  (17000, 16)
11089:ncbi:12  : OK  (17000, 16)
11089:ncbi:13  : OK  (17000, 16)
11089:ncbi:14  : OK  (17000, 16)
11089:ncbi:15  : OK  (17000, 16)
11089:ncbi:16  : OK  (17000, 16)
11089:ncbi:17  : OK  (17000, 16)
11089:ncbi:18  : OK  (17000, 16)
11089:ncbi:19  : OK  (17000, 16)
11089:ncbi:20  : OK  (17000, 16)
11089:ncbi:21  : OK  (17000, 16)
11089:ncbi:22  : OK  (17000, 16)
11089:ncbi:23  : OK  (17000, 16)
11089:ncbi:24  : OK  (17000, 16)
11089:ncbi:25  : OK  (17000, 16)
11089:ncbi:26  : OK  (17000, 16)
11089:ncbi:27  : OK  (17000, 16)
11089:ncbi:28  : OK  (17000, 16)
11089:ncbi:29  : OK  (17000, 16)
11089:ncbi:30  : OK  (17000, 16)
11089:ncbi

In [ ]:
for p in reports_to_modify:
    df = pd.read_csv(p)
    print(f"{p.name:30s}: {df.shape} {len(df.refseqid.unique())} unique refseqid")
    

11089_ncbi_44.csv             : (16779, 16) 1 unique refseqid
11089_ncbi_45.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_46.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_47.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_48.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_49.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_50.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_51.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_52.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_53.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_54.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_55.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_56.csv             : (16731, 16) 1 unique refseqid
11089_ncbi_57.csv             : (17000, 16) 1 unique refseqid
11089_ncbi_58.csv             : (16740, 16) 1 unique refseqid
11089_ncbi_59.csv             : (17000, 16) 1 unique refseqid
11089_nc

# End of Section